# Customer Spending


In [1]:
import pandas as pd
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [2]:
# Data preparation 
df = pd.read_csv("ShopSmart_data.csv")
df

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
0,46,620358.0,11.0,8,Silver,Beauty,Tier 1,Yes,16924.36
1,27,560886.0,13.0,7,Silver,Fashion,Tier 1,No,16467.52
2,52,866655.0,11.0,9,Bronze,Home,Tier 1,No,19224.61
3,55,816016.0,17.0,9,Silver,Home,Tier 2,Yes,20219.63
4,18,355468.0,14.0,0,Bronze,Home,Tier 3,Yes,7345.37
...,...,...,...,...,...,...,...,...,...
1995,47,557297.0,5.0,6,Gold,Home,Tier 1,No,17423.64
1996,46,1027242.0,11.0,12,Bronze,Fashion,Tier 3,No,19939.64
1997,40,677690.0,15.0,4,Bronze,Fashion,Tier 3,Yes,13397.79
1998,34,565410.0,11.0,8,Silver,Electronics,Tier 3,Yes,17682.11


EDA


In [3]:
df.head()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
0,46,620358.0,11.0,8,Silver,Beauty,Tier 1,Yes,16924.36
1,27,560886.0,13.0,7,Silver,Fashion,Tier 1,No,16467.52
2,52,866655.0,11.0,9,Bronze,Home,Tier 1,No,19224.61
3,55,816016.0,17.0,9,Silver,Home,Tier 2,Yes,20219.63
4,18,355468.0,14.0,0,Bronze,Home,Tier 3,Yes,7345.37


In [4]:
df.tail()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
1995,47,557297.0,5.0,6,Gold,Home,Tier 1,No,17423.64
1996,46,1027242.0,11.0,12,Bronze,Fashion,Tier 3,No,19939.64
1997,40,677690.0,15.0,4,Bronze,Fashion,Tier 3,Yes,13397.79
1998,34,565410.0,11.0,8,Silver,Electronics,Tier 3,Yes,17682.11
1999,65,397581.0,6.0,8,Bronze,Fashion,Tier 3,No,11165.42


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 2000 non-null   int64  
 1   Annual_Income       1984 non-null   float64
 2   Website_Visits      1984 non-null   float64
 3   Previous_Purchases  2000 non-null   int64  
 4   Membership_Type     1984 non-null   str    
 5   Preferred_Category  1984 non-null   str    
 6   City_Tier           2000 non-null   str    
 7   Discount_Used       2000 non-null   str    
 8   Amount_Spent        2000 non-null   float64
dtypes: float64(3), int64(2), str(4)
memory usage: 182.7 KB


In [6]:
df.isnull().sum()

Age                    0
Annual_Income         16
Website_Visits        16
Previous_Purchases     0
Membership_Type       16
Preferred_Category    16
City_Tier              0
Discount_Used          0
Amount_Spent           0
dtype: int64

In [7]:
avg_income = round(df["Annual_Income"].mean())
avg_web_visit = round(df["Website_Visits"].mean())
freq_membership=df["Membership_Type"].mode()[0]
freq_category =df["Preferred_Category"].mode()[0]


In [8]:
df["Annual_Income"] = df["Annual_Income"].fillna(avg_income)
df["Website_Visits"] = df["Website_Visits"].fillna(avg_web_visit)
df["Membership_Type"] = df["Membership_Type"].fillna(freq_membership)
df["Preferred_Category"] = df["Preferred_Category"].fillna(freq_category)


In [9]:
df.isnull().sum()

Age                   0
Annual_Income         0
Website_Visits        0
Previous_Purchases    0
Membership_Type       0
Preferred_Category    0
City_Tier             0
Discount_Used         0
Amount_Spent          0
dtype: int64

In [10]:
df[df.duplicated]

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent


In [11]:
df.describe()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Amount_Spent
count,2000.000000,2.000000e+03,2000.00000,2000.0000,2000.000000
mean,40.917500,7.525418e+05,9.95650,6.9655,17505.982405
std,12.780116,2.728578e+05,3.09134,2.6235,3866.061235
min,18.000000,1.800000e+05,2.00000,0.0000,6875.410000
25%,32.000000,5.605328e+05,8.00000,5.0000,14864.417500
50%,41.000000,7.520560e+05,10.00000,7.0000,17381.955000
75%,50.000000,9.346820e+05,12.00000,9.0000,19927.317500
max,65.000000,1.604606e+06,20.00000,16.0000,35371.200000


In [12]:
X = df.drop("Amount_Spent",axis = 1)
y = df["Amount_Spent"]

In [13]:
Xtrain,Xtest,ytrain,ytest = train_test_split(X,y,test_size=0.3,random_state=42)

In [14]:
num_cols = Xtrain.select_dtypes(include="number").columns
obj_cols = Xtrain.select_dtypes(include="object").columns
obj_cols

C:\Users\vishnu priya\AppData\Local\Temp\ipykernel_14600\4112315240.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = Xtrain.select_dtypes(include="object").columns


Index(['Membership_Type', 'Preferred_Category', 'City_Tier', 'Discount_Used'],
      dtype='str')

In [15]:
# multimodel pipeline
models = {
    "LinearRegression":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",LinearRegression())]),
    "Ridge":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",Ridge(alpha=50))]),
    "Lasso":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",Lasso(alpha=100))]),
    "KNN":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",KNeighborsRegressor(n_neighbors=6))]),
    "Decision Tree":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",DecisionTreeRegressor(max_depth=10))]),
    "Random Forest":Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),obj_cols)
                                                     ,("scler",StandardScaler(),num_cols)])),
                                 ("model",RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42))]),
                                 }

In [16]:
# multi model training 
model_perf={}
for model_name,pipeline in models.items():
    pipeline.fit(Xtrain,ytrain)
    ypred = pipeline.predict(Xtest)
    r2 = r2_score(ytest,ypred)
    mse = mean_squared_error(ytest,ypred)
    mae = mean_absolute_error(ytest,ypred)
    model_perf[model_name] = {"R2":r2,"MSE":mse,"MAE":mae}
model_perf

{'LinearRegression': {'R2': 0.9484114030748212,
  'MSE': 748773.1479047132,
  'MAE': 617.5717936465322},
 'Ridge': {'R2': 0.9455080053617377,
  'MSE': 790913.9769797483,
  'MAE': 637.8209549855577},
 'Lasso': {'R2': 0.9219361041019787,
  'MSE': 1133043.9778008086,
  'MAE': 821.426795372798},
 'KNN': {'R2': 0.8305839999172095,
  'MSE': 2458957.1969053196,
  'MAE': 1221.2485583333334},
 'Decision Tree': {'R2': 0.7610048200003547,
  'MSE': 3468851.3339862954,
  'MAE': 1438.6073478407557},
 'Random Forest': {'R2': 0.8286897791076457,
  'MSE': 2486450.514478208,
  'MAE': 1241.4181748366252}}

In [17]:
# model Performance table 
perf_df = pd.DataFrame(model_perf)
perf_df

,LinearRegression,Ridge,Lasso,KNN,Decision Tree,Random Forest
R2,0.948411,0.945508,9.219361e-01,8.305840e-01,7.610048e-01,8.286898e-01
MSE,748773.147905,790913.976980,1.133044e+06,2.458957e+06,3.468851e+06,2.486451e+06
MAE,617.571794,637.820955,8.214268e+02,1.221249e+03,1.438607e+03,1.241418e+03


In [18]:
# fetch the best model 
best_model = max(model_perf.items(), key = lambda t:t[1]['R2'])
print(best_model)
best_model_name = best_model[0]
best_model_name
best_piepline = models[best_model_name]
best_piepline



('LinearRegression', {'R2': 0.9484114030748212, 'MSE': 748773.1479047132, 'MAE': 617.5717936465322})


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Age','Annual_Income','Website_Visits',...,'Preferred_Category', 'City_Tier','Discount_Used']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('scler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thro

In [19]:
best_piepline.predict(Xtest)

array([14300.70821746, 18868.39803924, 21971.15353009, 18737.94405993,
       16436.42119351, 17202.76154403, 15377.20150439, 21325.65246883,
       15865.06206069, 17277.42577381, 13888.14976577, 16093.40582274,
       13677.31782594, 17415.88257148, 15663.74260122, 10125.80259628,
       18838.29969687, 22163.78213348, 15654.46952326, 15196.37727142,
       19751.23780891, 18291.28036664, 17704.59403663, 17981.41677218,
       15263.22950359, 20738.0649449 , 17682.11238067, 17959.57952301,
       18354.74055789, 12602.2609369 , 15750.38863006, 13783.45246416,
       28358.7905974 , 17406.7892587 , 16593.83373872, 15833.40324013,
       16347.84898046, 18796.29947389, 14010.96415369, 20601.09709545,
       16641.1926763 ,  9845.65563136, 16165.36509138, 23541.32722822,
       19370.07059054, 20790.03422243, 16708.21877095, 17124.57748792,
       19334.39449236, 23177.65065194, 18636.64185262, 12833.74443307,
        7984.53431687, 24789.60762942, 20556.17864924, 13783.75242183,
      

In [20]:
# model serialization 
import pickle
with open("best_model_pipeline.pkl","wb") as file:
    pickle.dump(best_piepline,file)


In [21]:
# save cleaned dataset
df.to_csv("cleaned_data.csv")

In [22]:
Xtrain.columns

Index(['Age', 'Annual_Income', 'Website_Visits', 'Previous_Purchases',
       'Membership_Type', 'Preferred_Category', 'City_Tier', 'Discount_Used'],
      dtype='str')

In [23]:
df["Discount_Used"].unique()

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str